# Project: Build a Transformer from scratch 🤖
The [[Mini-GPT on Danish]] was a *decoder-only* transformer. This is the **original** 2017 architecture from *Attention Is All You Need* ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762)): an **encoder** reads the input, a **decoder** writes the output, and **cross-attention** lets every output step look back at the input. That's how translation models (and Whisper, and T5) work.

**The task: translate messy Danish dates into ISO format.**
`fredag den 3. oktober 2026` → `2026-10-03` · `3/10-2026` → `2026-10-03` · `3. okt. 2026` → `2026-10-03`

Small enough to train in ~3 minutes on a CPU, hard enough to need attention (the model must *find* the year, map *oktober* → `10`, and reorder everything), and at the end you can **see** where it looks.

Topic: [[Transformers]], [[Attention Mechanism]] · guide note: [[Project - Build a Transformer from Scratch]]

You write the five core pieces: attention, multi-head attention, positional encoding, the decoder layer (with cross-attention) and greedy decoding. Everything else is provided.

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import math, random, datetime, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); random.seed(0); torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

MONTHS = ["januar", "februar", "marts", "april", "maj", "juni", "juli", "august", "september", "oktober", "november", "december"]
DAYS = ["mandag", "tirsdag", "onsdag", "torsdag", "fredag", "lørdag", "søndag"]
def danish(d):
    """One random Danish way of writing date d."""
    m, short = MONTHS[d.month - 1], MONTHS[d.month - 1][:3]
    return random.choice([f"{d.day}. {m} {d.year}", f"{DAYS[d.weekday()]} den {d.day}. {m} {d.year}", f"{d.day}/{d.month}-{d.year}",
                          f"{d.day:02d}.{d.month:02d}.{d.year}", f"{d.day}. {short}. {d.year}", f"d. {d.day}. {m} {d.year}",
                          f"{DAYS[d.weekday()][:3]}. {d.day}/{d.month} {d.year}"])
def random_date(): return datetime.date(1900, 1, 1) + datetime.timedelta(days=random.randrange(73048))   # 1900–2099
pairs = [(danish(d), d.isoformat()) for d in (random_date() for _ in range(30000))]
for s, t in pairs[:6]: print(f"{s:<32} → {t}")

PAD, BOS, EOS = 0, 1, 2
src_chars = sorted({c for s, _ in pairs for c in s}); tgt_chars = sorted({c for _, t in pairs for c in t})
s2i = {c: i + 3 for i, c in enumerate(src_chars)}; t2i = {c: i + 3 for i, c in enumerate(tgt_chars)}; i2t = {i: c for c, i in t2i.items()}
SRC_LEN, TGT_LEN = max(len(s) for s, _ in pairs) + 1, 10
def enc_src(s): return [s2i[c] for c in s] + [EOS] + [PAD] * (SRC_LEN - len(s) - 1)
def enc_tgt(t): return [t2i[c] for c in t]
X = torch.tensor([enc_src(s) for s, _ in pairs]); Y = torch.tensor([enc_tgt(t) for _, t in pairs])
X_tr, Y_tr, X_te, Y_te = X[:27000], Y[:27000], X[27000:], Y[27000:]
print(f"source vocab {len(s2i) + 3}, target vocab {len(t2i) + 3}, source length {SRC_LEN}, 27k train / 3k test")

## 1. Scaled dot-product attention
$\text{Attention}(Q,K,V) = \text{softmax}\!\big(\frac{QK^\top}{\sqrt{d_k}}\big)V$. `mask` is a boolean tensor (broadcastable to the score matrix) that is **True where attention is allowed**; set the other scores to $-\infty$ before the softmax. Return `(output, weights)`.

In [ ]:
def attention(q, k, v, mask=None):
    # TODO 1: scores, optional mask, softmax over the last dim, weighted sum of v
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_att():
    torch.manual_seed(1); q, k, v = torch.randn(3, 2, 4, 5, 8).unbind(0)
    mask = torch.tril(torch.ones(5, 5, dtype=torch.bool))
    out, w = attention(q, k, v, mask)
    return torch.allclose(out, F.scaled_dot_product_attention(q, k, v, attn_mask=mask), atol=1e-5) and torch.allclose(w.sum(-1), torch.ones(2, 4, 5)) and w[0, 0, 0, 1:].abs().max() == 0
check("attention", _t_att, "Must match F.scaled_dot_product_attention; rows of weights sum to 1; masked positions get weight 0.")

## 2. Multi-head attention
Project the queries (from `x_q`) and keys/values (from `x_kv`; the same tensor for self-attention, the encoder output for cross-attention) with `wq, wk, wv`, **split** $D$ into `h` heads of size $D/h$ → shape `(B, h, T, D/h)`, run `attention`, store the weights in `self.last_weights` (for plotting later), **merge** the heads back to `(B, Tq, D)` and apply `wo`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__(); self.h = h
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(d, d) for _ in range(4))
    def forward(self, x_q, x_kv, mask=None):
        # TODO 2: project, split heads, attention, merge heads, output projection
        raise NotImplementedError  # TODO 2

In [ ]:
def _t_mha():
    torch.manual_seed(2); m = MultiHeadAttention(16, 4); xq, xkv = torch.randn(3, 5, 16), torch.randn(3, 7, 16)
    out = m(xq, xkv)
    mha = nn.MultiheadAttention(16, 4, batch_first=True)
    with torch.no_grad():
        mha.in_proj_weight.copy_(torch.cat([m.wq.weight, m.wk.weight, m.wv.weight])); mha.in_proj_bias.copy_(torch.cat([m.wq.bias, m.wk.bias, m.wv.bias]))
        mha.out_proj.weight.copy_(m.wo.weight); mha.out_proj.bias.copy_(m.wo.bias)
    return out.shape == (3, 5, 16) and torch.allclose(out, mha(xq, xkv, xkv)[0], atol=1e-5) and m.last_weights.shape == (3, 4, 5, 7)
check("multi-head attention", _t_mha, "With the same weights your module must match torch.nn.MultiheadAttention; weights shape (B, h, Tq, Tk).")

## 3. Sinusoidal positional encoding
Attention is permutation-invariant, so we add position information: $PE_{p,2i} = \sin(p/10000^{2i/d})$, $PE_{p,2i+1} = \cos(p/10000^{2i/d})$. Return a `(max_len, d)` tensor.

In [ ]:
def positional_encoding(max_len, d):
    # TODO 3: sin on even columns, cos on odd columns
    raise NotImplementedError  # TODO 3

In [ ]:
def _t_pe():
    pe = positional_encoding(50, 16)
    return pe.shape == (50, 16) and torch.allclose(pe[0], torch.tensor([0.0, 1.0] * 8)) and abs(pe[3, 2] - math.sin(3 / 10000 ** (2 / 16))) < 1e-6 and abs(pe[3, 3] - math.cos(3 / 10000 ** (2 / 16))) < 1e-6
check("positional encoding", _t_pe, "Row 0 is [0, 1, 0, 1, …]; PE[3, 2] = sin(3 / 10000^(2/16)), PE[3, 3] = cos(…).")

## 4. The decoder layer (with cross-attention)
Pre-LayerNorm, three residual sub-layers:
1. **masked self-attention** over what has been written so far (`causal` mask),
2. **cross-attention**: queries from the decoder, keys/values from the encoder output `memory` (masking padded source positions with `src_mask`),
3. a position-wise **feed-forward** network.

`x = x + self_attn(ln1(x), ln1(x), causal)` etc. The encoder layer is given for comparison: it has no causal mask and no cross-attention.

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__(); self.attn = MultiHeadAttention(d, h); self.ff = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x, src_mask):
        x = x + self.attn(self.ln1(x), self.ln1(x), src_mask)
        return x + self.ff(self.ln2(x))

class DecoderLayer(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__(); self.self_attn, self.cross_attn = MultiHeadAttention(d, h), MultiHeadAttention(d, h)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d)); self.ln1, self.ln2, self.ln3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x, memory, causal, src_mask):
        # TODO 4: masked self-attention, cross-attention to memory, feed-forward – each pre-LN with a residual
        raise NotImplementedError  # TODO 4

class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d=64, h=4, ff=128, n_enc=2, n_dec=2):
        super().__init__()
        self.src_emb, self.tgt_emb = nn.Embedding(src_vocab, d), nn.Embedding(tgt_vocab, d)
        self.register_buffer("pe", positional_encoding(64, d))
        self.enc = nn.ModuleList([EncoderLayer(d, h, ff) for _ in range(n_enc)]); self.dec = nn.ModuleList([DecoderLayer(d, h, ff) for _ in range(n_dec)])
        self.ln_e, self.ln_d, self.out = nn.LayerNorm(d), nn.LayerNorm(d), nn.Linear(d, tgt_vocab); self.d = d
    def encode(self, src):
        src_mask = (src != PAD)[:, None, None, :]                                     # (B, 1, 1, Ts): don't attend to padding
        x = self.src_emb(src) + self.pe[: src.shape[1]]          # (no √d scaling: nn.Embedding already has std 1)
        for layer in self.enc: x = layer(x, src_mask)
        return self.ln_e(x), src_mask
    def decode(self, tgt_in, memory, src_mask):
        T = tgt_in.shape[1]; causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device))
        x = self.tgt_emb(tgt_in) + self.pe[:T]
        for layer in self.dec: x = layer(x, memory, causal, src_mask)
        return self.out(self.ln_d(x))
    def forward(self, src, tgt_in):
        memory, src_mask = self.encode(src); return self.decode(tgt_in, memory, src_mask)

In [ ]:
def _t_dec():
    torch.manual_seed(3); m = Transformer(len(s2i) + 3, len(t2i) + 3).eval()
    src = X_te[:2]; tgt = torch.cat([torch.full((2, 1), BOS), Y_te[:2, :-1]], 1)
    full = m(src, tgt); changed = tgt.clone(); changed[:, 5:] = 3
    part = m(src, changed)
    src2 = src.clone(); src2[:, 0] = s2i["9"] if src[0, 0] != s2i["9"] else s2i["1"]
    return full.shape == (2, 10, len(t2i) + 3) and torch.allclose(full[:, :5], part[:, :5], atol=1e-5) and not torch.allclose(m(src2, tgt), full)
check("decoder layer", _t_dec, "Outputs at positions < 5 must not change when later target tokens change (causal), but must change when the SOURCE changes (cross-attention).")

## 5. Greedy decoding
Encode once. Start every sequence with `BOS`; repeatedly run the decoder on what you have, take the **argmax** of the last position and append it, `max_len` times. Return the generated tokens **without** the BOS.

In [ ]:
def greedy_decode(model, src, max_len=TGT_LEN):
    # TODO 5: encode, then autoregressive argmax decoding
    raise NotImplementedError  # TODO 5

In [ ]:
def _t_greedy():
    torch.manual_seed(4); m = Transformer(len(s2i) + 3, len(t2i) + 3); out = greedy_decode(m, X_te[:3])
    with torch.no_grad():
        mem, sm = m.encode(X_te[:3]); first = m.decode(torch.full((3, 1), BOS), mem, sm)[:, -1].argmax(-1)
    return out.shape == (3, TGT_LEN) and torch.equal(out[:, 0], first)
check("greedy decoding", _t_greedy, "Output shape (B, 10) without BOS; the first token must be the argmax after feeding only BOS.")

## 6. Train it (teacher forcing)
The decoder input is the target shifted right (`BOS` + all but the last character); the loss compares its predictions with the real target. Adam with a short warm-up, batch 128, 6 epochs (≈ 45 s on a CPU).

In [ ]:
def to_text(ids): return "".join(i2t.get(int(i), "?") for i in ids)
EPOCHS = 6
if ready("attention", "multi-head attention", "positional encoding", "decoder layer", "greedy decoding"):
    torch.manual_seed(0); model = Transformer(len(s2i) + 3, len(t2i) + 3)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98))
    steps_per_epoch = len(X_tr) // 128; sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 300))
    print(f"{sum(p.numel() for p in model.parameters()) / 1e3:.0f}k parameters")
    for epoch in range(EPOCHS):
        model.train(); perm = torch.randperm(len(X_tr)); tot = 0
        for i in range(steps_per_epoch):
            idx = perm[i * 128:(i + 1) * 128]; src, tgt = X_tr[idx], Y_tr[idx]
            tgt_in = torch.cat([torch.full((len(idx), 1), BOS), tgt[:, :-1]], 1)
            loss = F.cross_entropy(model(src, tgt_in).reshape(-1, len(t2i) + 3), tgt.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item()
        pred = greedy_decode(model, X_te[:1000]); acc = (pred == Y_te[:1000]).all(1).float().mean().item()
        print(f"epoch {epoch + 1}: loss {tot / steps_per_epoch:.4f}, exact-match on 1000 test dates {acc:.1%}")

In [ ]:
if ready("attention", "multi-head attention", "positional encoding", "decoder layer", "greedy decoding"):
    pred = greedy_decode(model, X_te); exact = (pred == Y_te).all(1).float().mean().item()
    print(f"exact match on all 3000 test dates: {exact:.2%}")
    for s in ["fredag den 3. oktober 2026", "24/12-1999", "1. jan. 2000", "søn. 29/2 2004", "d. 17. maj 1950"]:
        print(f"{s:<28} → {to_text(greedy_decode(model, torch.tensor([enc_src(s)]))[0])}")
    wrong = (pred != Y_te).any(1).nonzero().flatten()[:5]
    for i in wrong: print("  miss:", pairs[27000 + int(i)][0], "→", to_text(pred[i]), "(want", pairs[27000 + int(i)][1] + ")")
    check("translates dates", lambda: exact > 0.95, "Expect > 95 % exact match after 6 epochs. Check the residuals and the masks.")

> [!warning] A bug I hit while building this
> The paper multiplies embeddings by $\sqrt{d}$. With PyTorch's `nn.Embedding` (initialised with std 1) that makes the embeddings ~8× larger than the positional encoding, the model can barely tell positions apart, and it writes *2062* for *2026*. Accuracy after 15 epochs: 65 %. Without the scaling: 100 % after 6. Try it: put `* math.sqrt(self.d)` back in `encode`/`decode`.

## 7. Look inside: cross-attention
Which input characters does the decoder look at while writing each output character? (last decoder layer, averaged over heads)

In [ ]:
if ready("attention", "multi-head attention", "positional encoding", "decoder layer", "greedy decoding"):
    s = "fredag den 3. oktober 2026"; src = torch.tensor([enc_src(s)]); out = greedy_decode(model, src)
    with torch.no_grad():
        mem, sm = model.encode(src); model.decode(torch.cat([torch.full((1, 1), BOS), out[:, :-1]], 1), mem, sm)
    w = model.dec[-1].cross_attn.last_weights[0].mean(0)[:, : len(s) + 1].numpy()
    plt.figure(figsize=(10, 3.5)); plt.imshow(w, cmap="viridis", aspect="auto")
    plt.xticks(range(len(s) + 1), list(s) + ["<eos>"]); plt.yticks(range(TGT_LEN), list(to_text(out[0])))
    plt.xlabel("Danish input"); plt.ylabel("ISO output"); plt.title("cross-attention: where the decoder looks"); plt.show()
    print("→ The year digits attend to '2026', the month digits to 'oktober', the day to '3'. Nobody told it to align them.")

<details><summary>▶ Solution</summary>

```python
def attention(q, k, v, mask=None):
    d = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ v, weights
```

```python
    def forward(self, x_q, x_kv, mask=None):   # MultiHeadAttention
        B, Tq, D = x_q.shape; Tk = x_kv.shape[1]
        split = lambda t, T: t.view(B, T, self.h, D // self.h).transpose(1, 2)
        q, k, v = split(self.wq(x_q), Tq), split(self.wk(x_kv), Tk), split(self.wv(x_kv), Tk)
        out, self.last_weights = attention(q, k, v, mask)
        return self.wo(out.transpose(1, 2).reshape(B, Tq, D))
```

```python
def positional_encoding(max_len, d):
    pos = torch.arange(max_len).unsqueeze(1).float()
    i = torch.arange(0, d, 2).float()
    angle = pos / (10000 ** (i / d))
    pe = torch.zeros(max_len, d)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe
```

```python
    def forward(self, x, memory, causal, src_mask):   # DecoderLayer
        x = x + self.self_attn(self.ln1(x), self.ln1(x), causal)
        x = x + self.cross_attn(self.ln2(x), memory, src_mask)
        x = x + self.ff(self.ln3(x))
        return x
```

```python
def greedy_decode(model, src, max_len=TGT_LEN):
    model.eval()
    with torch.no_grad():
        memory, src_mask = model.encode(src)
        out = torch.full((src.shape[0], 1), BOS, dtype=torch.long)
        for _ in range(max_len):
            logits = model.decode(out, memory, src_mask)[:, -1]
            out = torch.cat([out, logits.argmax(-1, keepdim=True)], dim=1)
    return out[:, 1:]
```
</details>

## Make it yours (stretch goals)
- **Harder inputs:** add written-out days (*tredje oktober*), typos, or two-digit years. Where does it break first?
- **Ablations:** remove positional encoding, or use 1 head instead of 4. What happens to accuracy and to the attention map?
- **Beam search** instead of greedy decoding.
- **Real translation:** train on Danish↔English sentence pairs (e.g. the Tatoeba corpus) with the same code and a BPE tokenizer from [[Project - BPE Tokenizer on My Notes]].
- **Compare:** solve the same task with a GRU encoder–decoder without attention ([[RNN and LSTM]]). Accuracy vs. input length?